# Carga de Modelo Estrella hacia Databricks

Este notebook se ejecuta localmente. Debido a restricciones de red/proxy al momento de escribir archivos físicos hacia los endpoints subyacentes de la nube (Volúmenes), **usaremos Inserción SQL Nativa**. Leemos los archivos con `pandas`, eliminamos filas duplicadas por llave primaria, construimos lotes (*batches*) grandes, realizamos la inserción directamente sobre los staging, completamos el cruce `MERGE INTO`, y finalmente limpiamos la tabla temporal.

In [1]:
%pip install databricks-sql-connector python-dotenv pandas -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from databricks import sql

# 1. Cargar credenciales
load_dotenv()

HOST = os.getenv('DATABRICKS_HOST')
TOKEN = os.getenv('DATABRICKS_TOKEN')
HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH') 

CATALOG = 'client_segmentation'
SCHEMA = 'tablas'

print(f"☑️ Credenciales Listas. Motor SQL: {HTTP_PATH}")

☑️ Credenciales Listas. Motor SQL: /sql/1.0/warehouses/4281b68857863e33


### Función de Ayuda e Inicialización DDL

In [3]:
def execute_query(query):
    with sql.connect(server_hostname=HOST.replace('https://', ''), http_path=HTTP_PATH, access_token=TOKEN) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            try:
                return cursor.fetchall()
            except:
                return None

# Preparación del esquema
execute_query(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Creación de las Tablas Finales
ddl_customer = f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.dim_customer (
    customer_id STRING,
    customer_since DATE,
    location STRING,
    customer_type STRING
)
"""
execute_query(ddl_customer)

ddl_product = f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.dim_product (
    product_id STRING,
    product_name STRING,
    brand STRING,
    product_category STRING,
    product_subcategory STRING,
    fuel_type STRING,
    vehicle_segment STRING,
    is_vehicle INTEGER
)
"""
execute_query(ddl_product)

ddl_sales = f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.fact_sales (
    transaction_id STRING,
    customer_id STRING,
    product_id STRING,
    date DATE,
    quantity INTEGER,
    unit_price DOUBLE,
    discount DOUBLE,
    total_amount DOUBLE,
    product_category STRING,
    channel STRING,
    payment_method STRING
)
"""
execute_query(ddl_sales)
print("☑️ Tablas finales DDL inicializadas")

☑️ Tablas finales DDL inicializadas


### Carga de Datos Vía Pandas y BATCH Inserts
Transformamos el cargado a lotes de 3000 filas insertadas de una por un motor SQL puro.

In [4]:
def format_sql_value(val):
    if pd.isna(val) or val == '':
        return "NULL"
    if isinstance(val, (int, float, np.integer, np.floating)):
        return str(val)
    # Limpiamos comillas simples que puedan romper el query
    str_val = str(val).replace("'", "''")
    return f"'{str_val}'"

files_mapping = {
    "dim_customer.csv": ("dim_customer", "customer_id"),
    "dim_product.csv": ("dim_product", "product_id"),
    "fact_sales.csv": ("fact_sales", "transaction_id")
}

batch_size = 3000

for file_name, (table, pk) in files_mapping.items():
    table_staging = f"{CATALOG}.{SCHEMA}.{table}_staging"
    table_final = f"{CATALOG}.{SCHEMA}.{table}"
    
    print(f"\nProcesando {table} desde {file_name}...")
    
    # 1. Crear staging vacía y limpiarla
    execute_query(f"CREATE TABLE IF NOT EXISTS {table_staging} LIKE {table_final}")
    execute_query(f"TRUNCATE TABLE {table_staging}")
    
    # 2. Leer con Pandas localmente y Deduplicar
    df = pd.read_csv(file_name)
    df = df.drop_duplicates(subset=[pk], keep='first')
    cols = ", ".join(df.columns)
    
    # 3. Lógica Batch de SQL puro
    total_rows = len(df)
    for i in range(0, total_rows, batch_size):
        batch_df = df.iloc[i:i+batch_size]
        values_list = []
        
        for row in batch_df.itertuples(index=False, name=None):
            row_vals = [format_sql_value(v) for v in row]
            values_list.append("(" + ", ".join(row_vals) + ")")
            
        all_values_str = ", ".join(values_list)
        insert_query = f"INSERT INTO {table_staging} ({cols}) VALUES {all_values_str}"
        execute_query(insert_query)
    
    print(f" ☑️ {total_rows} filas limpias subidas directas al Staging")
    
    # 4. Hacer Merge into en la final
    merge_query = f"""
    MERGE INTO {table_final} target
    USING {table_staging} source
    ON target.{pk} = source.{pk}
    WHEN NOT MATCHED THEN
      INSERT *
    """
    execute_query(merge_query)
    print(f" ☑️ MERGE INTO de {table} completado con éxito.")
    
    # 5. Limpieza de Tabla Staging
    execute_query(f"DROP TABLE IF EXISTS {table_staging}")
    print(f" 🗑️ Tabla staging temporal {table_staging} borrada del entorno (Limpieza finalizada).")

print("\n🚀 PIPELINE TERMINADO COMPLETO 🚀")


Procesando dim_customer desde dim_customer.csv...
 ☑️ 5000 filas limpias subidas directas al Staging
 ☑️ MERGE INTO de dim_customer completado con éxito.
 🗑️ Tabla staging temporal client_segmentation.tablas.dim_customer_staging borrada del entorno (Limpieza finalizada).

Procesando dim_product desde dim_product.csv...
 ☑️ 300 filas limpias subidas directas al Staging
 ☑️ MERGE INTO de dim_product completado con éxito.
 🗑️ Tabla staging temporal client_segmentation.tablas.dim_product_staging borrada del entorno (Limpieza finalizada).

Procesando fact_sales desde fact_sales.csv...
 ☑️ 42500 filas limpias subidas directas al Staging
 ☑️ MERGE INTO de fact_sales completado con éxito.
 🗑️ Tabla staging temporal client_segmentation.tablas.fact_sales_staging borrada del entorno (Limpieza finalizada).

🚀 PIPELINE TERMINADO COMPLETO 🚀
